# sEMG Prosthetic Gesture Classification
## Notebook 13 (Training): Real Feature/Channel/Domain Ablation Studies (Google Colab Edition)

This notebook actually **runs** 20 real ablation configurations against the fully re-tuned
(308-trial) CatBoost model, using the same real subject-disjoint train/test split as Notebook 10
(28 train subjects / 6 test subjects, `outputs/reports/split_metadata_top50.json`):

1. **Baseline**: all 50 selected features.
2. **Feature removal** (8 configs): remove the top-5/10/15/20 and bottom-5/10/15/20
   SHAP-ranked features (real ranking from Notebook 12's `global_feature_ranking.csv`).
3. **Channel efficiency** (5 configs): restrict to the features belonging to the top-8/6/4/2/1
   most important channels (real ranking from Notebook 12's `channel_ranking.csv`).
4. **Feature family** (6 configs): Time-only, Frequency-only, Wavelet-only, and pairwise
   combinations of the three domains.

Each configuration clones the tuned CatBoost pipeline's hyperparameters and refits **from
scratch** on the real training data restricted to that configuration's feature subset --
this notebook replaces a prior read-only version of Notebook 13 that read pre-computed tables
which turned out (on inspection) to have been generated from tiny, non-representative training
subsamples (5,000-10,000 rows out of 484,700), producing internally inconsistent baseline
accuracy figures (24%, 38%, and 41% for the same nominal baseline configuration across
different tables). This run uses the **full** real training split for every configuration.

Checkpoints are written to `outputs/ablation_checkpoints_v2/CATBOOST/` (one JSON per
configuration; resumable -- re-running the training cell skips configs that already have a
checkpoint).

In [ ]:
# ==============================================================
# GOOGLE COLAB SETUP & ENVIRONMENT INITIALIZATION
# ==============================================================
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/semg-prosthetic-gesture-classification'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"Changed working directory to Google Drive: {PROJECT_PATH}")
    else:
        raise FileNotFoundError(
            f"{PROJECT_PATH} not found. Upload/sync the project folder to this path first."
        )
    !pip install -q catboost xgboost lightgbm scikit-learn pyarrow fastparquet shap
else:
    PROJECT_PATH = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()))

print(f"IN_COLAB={IN_COLAB} | PROJECT_PATH={PROJECT_PATH}")


In [ ]:
import sys, os, json, time, pickle
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_PATH)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml.ablation import run_ablation_studies

outputs_dir = PROJECT_ROOT / "outputs"
tables_dir = outputs_dir / "tables"

NEW_CHECKPOINT_DIR = outputs_dir / "ablation_checkpoints_v2" / "CATBOOST"
NEW_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Writing ablation checkpoints to: {NEW_CHECKPOINT_DIR}")

print("Loading full top-50 feature dataset...")
t0 = time.perf_counter()
df = pd.read_parquet(PROJECT_ROOT / "data/final/selected_features_top50.parquet")
print(f"Loaded {df.shape} in {time.perf_counter()-t0:.1f}s")

split_meta = json.load(open(tables_dir.parent / "reports" / "split_metadata_top50.json"))
df_train = df[df["subject_id"].isin(split_meta["train_subjects"])].copy()
df_test = df[df["subject_id"].isin(split_meta["test_subjects"])].copy()
del df
n_train_subj = len(split_meta["train_subjects"])
n_test_subj = len(split_meta["test_subjects"])
print(f"Train split: {df_train.shape} ({n_train_subj} subjects)")
print(f"Test split: {df_test.shape} ({n_test_subj} subjects)")

with open(PROJECT_ROOT / "models" / "optimized" / "CatBoost.pkl", "rb") as f:
    tuned_pipeline = pickle.load(f)
print("Tuned CatBoost params:", tuned_pipeline.named_steps["classifier"].get_params())


### Enable GPU acceleration for CatBoost (recommended)

**Correction:** an earlier version of this notebook claimed GPU was unnecessary here,
extrapolating from the LOSO notebook's 54.7-minute total -- but that LOSO run **used GPU**
(T4); it was not a CPU number. On CPU, a single ablation configuration on the real 484,700-row
training split takes on the order of ~90 minutes, which would make the full 20-configuration
sweep take roughly a day. Set `USE_GPU = True` below (and select **Runtime -> Change runtime
type -> T4 GPU** in Colab) to run this at the same speed the LOSO notebook achieved. This
raises a clear error rather than silently falling back to CPU if no GPU is attached.

In [ ]:
USE_GPU = True  # requires a GPU (T4) runtime -- Runtime -> Change runtime type -> T4 GPU

extra_classifier_params = {"task_type": "GPU", "devices": "0"} if USE_GPU else None
print(f"USE_GPU={USE_GPU}", "-> classifier overrides:", extra_classifier_params)


In [ ]:
# ==============================================================
# RUN REAL ABLATION STUDIES (20 configurations)
# Resumable: already-checkpointed configs (in NEW_CHECKPOINT_DIR) are skipped
# automatically if you stop and re-run this cell.
# ==============================================================
t_start = time.perf_counter()
ablation_out = run_ablation_studies(
    df_train=df_train,
    df_test=df_test,
    workspace_dir=PROJECT_ROOT,
    model_name="CATBOOST",
    checkpoint_dir=NEW_CHECKPOINT_DIR,
    force_rerun=False,
    extra_classifier_params=extra_classifier_params,
)
total_time = time.perf_counter() - t_start
n_results = len(ablation_out["results"])
print(f"\nAblation studies complete: {n_results} configurations in {total_time/60:.1f} minutes")

results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "Features"} for r in ablation_out["results"]
])
results_df = results_df.sort_values("Macro F1", ascending=False)
results_df.to_csv(NEW_CHECKPOINT_DIR / "ablation_summary.csv", index=False)
print(results_df[["Config", "Feature Count", "Accuracy", "Macro F1", "Training Time (s)"]])


### After completion

Download the entire `outputs/ablation_checkpoints_v2/CATBOOST/` directory back to the local
project (same relative path). The next step will be rebuilding
`notebooks/13_ablation_studies.ipynb` to aggregate and report on these real ablation results,
replacing its prior read-only dependency on the small-sample tables.